# Feature Engineering:
this part will be dedicated to feature engineering and validation of the newly generate features, using the following Filtering techniques:
1. Correlation Matrix with Heatmap
2. Variance Threshold
3. Variance Inflation Factor (VIF)

Embedded Methods used:
1. Tree-based feature Importance

Wrapper Methods used:
1. Recursive Feature Elimination (RFE)  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot style
sns.set_style("whitegrid")

# directory paths for exporting and creating data
metadata_path = './cleaned_metadata.csv'
impedance_final_path = './processed_data/cleaned_impedance_metadata.csv'
soh_table_path = './processed_data/soh_calculations.csv'
feature_aggregation_path = './processed_data/feature_aggregation.csv'

# 1. Data Loading

In [2]:
# Load metadata
metadata = pd.read_csv(metadata_path)
print(metadata.info())
metadata.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6493 entries, 0 to 6492
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   type                 6493 non-null   object 
 1   start_time           6493 non-null   object 
 2   ambient_temperature  6493 non-null   int64  
 3   battery_id           6493 non-null   object 
 4   test_id              6493 non-null   int64  
 5   uid                  6493 non-null   int64  
 6   filename             6493 non-null   object 
 7   Capacity             2249 non-null   float64
 8   Re                   1734 non-null   float64
 9   Rct                  1705 non-null   float64
dtypes: float64(3), int64(3), object(4)
memory usage: 507.4+ KB
None


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,2010-07-21 15:00:35.093,4,B0047,0,1,00001.csv,1.674305,NaN,NaN
1,impedance,2010-07-21 16:53:45.968,24,B0047,1,2,00002.csv,NaN,0.056058,0.200970
2,charge,2010-07-21 17:25:40.671,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,2010-07-21 20:31:05.000,24,B0047,3,4,00004.csv,NaN,0.053192,0.164734
4,discharge,2010-07-21 21:02:56.984,4,B0047,4,5,00005.csv,1.524366,NaN,NaN


### Initial Capacity Extraction:


In [3]:
discharge_df = metadata[metadata['type'] == 'discharge'].copy()
discharge_df['Capacity'] = pd.to_numeric(discharge_df['Capacity'], errors='coerce')
# Find C_init (Initial Capacity) for every Battery
# We sort by time to ensure we grab the very first cycle
discharge_df = discharge_df.sort_values(by=['battery_id', 'start_time'])

# Group by battery and take the first Capacity value
c_init_map = discharge_df.groupby('battery_id')['Capacity'].first()

print("Initial Capacities (C_init) per Battery:")
print(c_init_map)

Initial Capacities (C_init) per Battery:
battery_id
B0005    1.856487
B0006    2.035338
B0007    1.891052
B0018    1.855005
B0029    1.697507
B0030    1.656071
B0031    1.666675
B0032    1.704864
B0033    1.652351
B0034    1.524531
B0036    1.801101
B0038    1.779742
B0040    1.707514
B0042    1.728713
B0043    1.713783
B0044    1.686526
B0045    1.081979
B0046    1.728239
B0047    1.674305
B0048    1.657996
B0053    1.121790
B0054    1.075445
B0055    1.206764
B0056    1.263960
Name: Capacity, dtype: float64


## Feature dictionary (what each feature represents + why it matters)
This section explains every engineered feature we compute from the per-cycle time-series logs in `data/` and how it relates to degradation (SOH/RUL).

### A) Discharge-cycle features (computed from discharge `data/<file>.csv`)
These come from columns: `Time` (s), `Voltage_measured` (V), `Current_measured` (A), `Temperature_measured` (°C).  
We convert discharge current to a **positive magnitude** using $I_{dis}(t)=\max(-I(t),0)$ (because discharge is typically logged as negative current in this dataset).

**Time / activity**
- `duration_s`: $t_{end}-t_{start}$ in seconds. Longer/shorter discharge at the same protocol can reflect usable capacity and polarization effects.
- `frac_active_discharge`: fraction of samples where $I_{dis}(t) \ge 0.05\,A$. Helps detect cycles that are mostly rest/pulses (often corrupted or not a true discharge), and provides context for other integrals.
- `dis_time`: the last recorded time value (kept for backward compatibility). Prefer `duration_s` if time doesn’t start at 0.

**Voltage landmarks (knee / curve shift)**
- `time_to_3v8`: first time when $V(t) \le 3.8\,V$. Captures early voltage sag; tends to happen earlier as impedance increases and SOH drops.
- `time_to_3v6`: first time when $V(t) \le 3.6\,V$. A proxy for the “knee point shift” near end-of-discharge; very sensitive to aging.
- `volt_at_600s`: $V(t)$ at 600 s (10 minutes). A standardized snapshot of the curve useful when discharge durations vary; typically decreases with aging at fixed load.
- `v_t10`, `v_t50`, `v_t90`: voltage at 10/50/90% of the cycle duration. These summarize curve shape with 3 numbers (early/mid/late). Aging changes the curve shape even when end voltage is similar.

**Early resistance proxy**
- `volt_drop`: $V_{start}-V_{load}$ using the first active-discharge point (avoids the “rest” sample). Larger drop often indicates higher internal resistance and polarization.
- `r0_ohm`: $R_0 \approx \frac{V_{start}-V_{load}}{I_{load}}$ (Ohm). A simple physics-motivated internal resistance estimate; resistance generally increases with aging.

**Integrals (throughput / losses)**
- `cap_int_ah`: Coulomb-counted discharge capacity in Ah: $$C_{int}=\frac{1}{3600}\int I_{dis}(t)\,dt$$ Useful as an independent capacity estimate derived from raw logs (quality check and sometimes predictive when metadata capacity is noisy).
- `energy_int_wh`: Delivered energy in Wh (approx): $$E_{int}=\frac{1}{3600}\int V(t)\,I_{dis}(t)\,dt$$ Energy often declines with aging due to voltage sag even if capacity declines slowly.

**Signal statistics (compact summaries)**
- `v_min`, `v_max`, `v_mean`, `v_std`: min/max/mean/std of measured voltage within the cycle. Provide the voltage range and variability; voltage sag increases with aging.
- `i_dis_min`, `i_dis_max`, `i_dis_mean`, `i_dis_std`: stats of discharge current magnitude $I_{dis}$. Helps control for slight protocol variability; also flags abnormal cycles.
- `temp_min`, `temp_max`, `temp_mean`, `temp_std`: stats of temperature during discharge. Temperature affects kinetics and measured capacity; it’s also a proxy for resistive heating.
- `temp_rise`: $T_{max}-T_{min}$ during discharge. Higher rise at same load suggests higher losses (often linked to increased resistance).
- `temp_slope`: $(T_{end}-T_{start})/duration_s$ (°C/s). Similar idea to `temp_rise` but normalized by time.

**Dynamics (how fast voltage changes)**
- `dvdt_min`, `dvdt_max`, `dvdt_mean`, `dvdt_std`: stats of $dV/dt$ computed from successive samples. Aging can increase polarization, changing the typical slope profile; these features capture that without needing full curves.

### B) Charge-cycle features (computed from charge `data/<file>.csv`)
These come from the same core columns as discharge (and optionally `Current_charge` / `Voltage_charge` if present). We convert charge current to a positive magnitude using $I_{ch}(t)=\max(I(t),0)$.

**Time / phase behavior (CC-CV proxy)**
- `chg_duration_s`: total charging duration (s). Can change with aging under CCCV charging (CV tail behavior).
- `chg_time_to_4p0`: first time when $V(t) \ge 4.0\,V$. Captures early charge dynamics / polarization.
- `chg_time_to_4p2`: first time when $V(t) \ge 4.2\,V$. Indicates when the cell reaches the high-voltage region; often shifts with health and temperature.
- `chg_cv_time_s`: time spent at high voltage (default: $V \ge 4.18\,V$). A crude CV-tail duration proxy; CV tail often changes as impedance increases.

**Integrals (charge acceptance / energy input)**
- `chg_q_ah`: Coulomb-counted charge throughput in Ah: $$Q_{ch}=\frac{1}{3600}\int I_{ch}(t)\,dt$$ Can reflect charge acceptance and protocol consistency.
- `chg_energy_wh`: charging energy in Wh: $$E_{ch}=\frac{1}{3600}\int V(t)\,I_{ch}(t)\,dt$$ Captures energy input; combined with discharge energy can indicate efficiency trends.

**End-of-charge tail**
- `chg_v_end`: last measured charge voltage. Sanity/protocol check (should be near cutoff).
- `chg_i_end`: last charge current magnitude (tail current). In CCCV, tail current behavior is informative about resistance and aging.

**Thermal response**
- `chg_temp_rise`: $T_{max}-T_{min}$ during charge. Increased heating at similar protocol can indicate higher losses/impedance.
- `chg_temp_max`: maximum measured temperature during charge.
- `chg_v_*`, `chg_i_*`, `chg_temp_*`: min/max/mean/std of charge voltage/current/temperature within the cycle.

**Command tracking (only if columns exist)**
- `chg_i_cmd_mae`: mean absolute error between measured and commanded current (A). Helps detect abnormal control/measurement behavior.
- `chg_v_cmd_mae`: mean absolute error between measured and commanded voltage (V). Same idea for voltage control.

### C) Using `metadata.csv` (why we keep it)
We keep metadata columns because they are critical context for modeling and for correct alignment across cycle types:
- `battery_id`: identifies the unit; used for grouping, plotting, and (most importantly) train/test splitting by battery.
- `test_id`: cycle index used to align charge/discharge/impedance and to build lag/rolling features later.
- `ambient_temperature`: strong confounder (temperature changes the curves and capacity). It should be included as a feature or used for stratification.
- `Capacity` (discharge only): usually a target/label (SOH uses it). Avoid using it as an input feature when predicting SOH for the same cycle (label leakage).

In [4]:
from pathlib import Path

DATA_DIR = Path("data")

def _safe_read_cycle_csv(filename: str) -> pd.DataFrame | None:
    """Read a per-cycle CSV from data/ and normalize column names."""
    path = DATA_DIR / str(filename)
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    df.columns = [c.strip() for c in df.columns]
    return df


def _to_num(series: pd.Series) -> np.ndarray:
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)


def _series_stats(x: np.ndarray, prefix: str) -> dict:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {f"{prefix}_min": np.nan, f"{prefix}_max": np.nan, f"{prefix}_mean": np.nan, f"{prefix}_std": np.nan}
    return {
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
    }


def _value_at_time(t: np.ndarray, y: np.ndarray, target_t: float) -> float:
    """Return y at first index where t>=target_t (NaN if not possible)."""
    m = np.isfinite(t) & np.isfinite(y)
    t2, y2 = t[m], y[m]
    if t2.size == 0:
        return np.nan
    order = np.argsort(t2)
    t2, y2 = t2[order], y2[order]
    idx = np.searchsorted(t2, target_t, side="left")
    if idx >= t2.size:
        return float(y2[-1])
    return float(y2[idx])


def _time_to_voltage(t: np.ndarray, v: np.ndarray, threshold: float, direction: str) -> float:
    """Time when V crosses a threshold.
    direction: 'below' (discharge) or 'above' (charge).
    """
    m = np.isfinite(t) & np.isfinite(v)
    t2, v2 = t[m], v[m]
    if t2.size == 0:
        return np.nan
    order = np.argsort(t2)
    t2, v2 = t2[order], v2[order]
    if direction == "below":
        hit = np.where(v2 <= threshold)[0]
    else:
        hit = np.where(v2 >= threshold)[0]
    return float(t2[hit[0]]) if hit.size else float(t2[-1])



def extract_discharge_features(filename: str, current_capacity, c_init) -> dict | None:
    """Extract richer discharge features from a discharge cycle log file."""
    df = _safe_read_cycle_csv(filename)
    if df is None or df.empty:
        return None
    needed = {"Time", "Voltage_measured", "Current_measured", "Temperature_measured"}
    if not needed.issubset(set(df.columns)):
        return None
    t = _to_num(df["Time"])
    v = _to_num(df["Voltage_measured"])
    i = _to_num(df["Current_measured"])
    temp = _to_num(df["Temperature_measured"])

    # Basic time cleaning
    m = np.isfinite(t) & np.isfinite(v) & np.isfinite(i)
    if m.sum() < 5:
        return None
    t, v, i, temp = t[m], v[m], i[m], temp[m]
    order = np.argsort(t)
    t, v, i, temp = t[order], v[order], i[order], temp[order]
    duration = float(t[-1] - t[0]) if t.size else np.nan

    # Discharge current magnitude (in this dataset discharge is usually negative)
    i_dis = -np.minimum(i, 0.0)  # positive magnitude during discharge
    active = i_dis >= 0.05
    frac_active = float(active.mean()) if active.size else np.nan

    # Coulomb-counted capacity + energy (approx)
    cap_ah = float(np.trapezoid(i_dis, t) / 3600.0) if np.isfinite(duration) and duration > 0 else np.nan
    energy_wh = float(np.trapezoid(v * i_dis, t) / 3600.0) if np.isfinite(duration) and duration > 0 else np.nan

    # Existing features (kept)
    dis_time = float(t[-1]) if t.size else np.nan
    time_to_3v6 = _time_to_voltage(t, v, 3.6, direction="below")
    volt_at_600s = _value_at_time(t, v, 600.0)
    temp_slope = float((temp[-1] - temp[0]) / duration) if np.isfinite(duration) and duration > 0 else np.nan
    temp_max = float(np.nanmax(temp)) if np.isfinite(temp).any() else np.nan

    # Improved initial drop / resistance proxy
    v_start = float(v[0])
    # first active-discharge point (avoid rest sample)
    idx_active = np.where(active)[0]
    if idx_active.size:
        j = int(idx_active[0])
        v_load = float(v[j])
        i_load = float(i_dis[j])
    else:
        v_load = float(v[1]) if v.size > 1 else float(v[0])
        i_load = float(i_dis[1]) if i_dis.size > 1 else float(i_dis[0])
    volt_drop = float(v_start - v_load)
    r0_ohm = float(volt_drop / i_load) if np.isfinite(i_load) and i_load > 1e-6 else np.nan
    if c_init and c_init > 0:
        soh = current_capacity / c_init
    else:
        soh = 0
    # Extra discharge features
    feats = {
        "filename": filename,
        "dis_time": dis_time,
        "duration_s": duration,
        "frac_active_discharge": frac_active,
        "time_to_3v6": time_to_3v6,
        "time_to_3v8": _time_to_voltage(t, v, 3.8, direction="below"),
        "volt_drop": volt_drop,
        "r0_ohm": r0_ohm,
        "volt_at_600s": volt_at_600s,
        "cap_int": c_init,
        "energy_int_wh": energy_wh,
        "temp_slope": temp_slope,
        "temp_rise": float(np.nanmax(temp) - np.nanmin(temp)) if np.isfinite(temp).any() else np.nan,
        "temp_max": temp_max,
        "current_cap": current_capacity,
        "SOH": soh,
    }
    feats.update(_series_stats(v, "v"))
    feats.update(_series_stats(i_dis, "i_dis"))
    feats.update(_series_stats(temp, "temp"))

    # Shape features: voltage at time percentiles
    if t.size:
        t0 = float(t[0])
        t1 = float(t[-1])
        for p in [0.1, 0.5, 0.9]:
            feats[f"v_t{int(p*100)}"] = _value_at_time(t, v, t0 + p * (t1 - t0))
    else:
        feats["v_t10"] = feats["v_t50"] = feats["v_t90"] = np.nan

    # dV/dt stats (rough)
    dt = np.diff(t)
    dv = np.diff(v)
    with np.errstate(divide="ignore", invalid="ignore"):
        dvdt = np.where(dt > 0, dv / dt, np.nan)
    feats.update(_series_stats(dvdt, "dvdt"))

    return feats


def extract_charge_features(filename: str) -> dict | None:
    """Extract CCCV-like charge features from a charge cycle log file."""
    df = _safe_read_cycle_csv(filename)
    if df is None or df.empty:
        return None
    needed = {"Time", "Voltage_measured", "Current_measured", "Temperature_measured"}
    if not needed.issubset(set(df.columns)):
        return None
    t = _to_num(df["Time"])
    v = _to_num(df["Voltage_measured"])
    i = _to_num(df["Current_measured"])
    temp = _to_num(df["Temperature_measured"])

    m = np.isfinite(t) & np.isfinite(v) & np.isfinite(i)
    if m.sum() < 5:
        return None
    t, v, i, temp = t[m], v[m], i[m], temp[m]
    order = np.argsort(t)
    t, v, i, temp = t[order], v[order], i[order], temp[order]
    duration = float(t[-1] - t[0]) if t.size else np.nan

    # Charge current magnitude (assume positive during charge)
    i_ch = np.maximum(i, 0.0)
    active = i_ch >= 0.05
    frac_active = float(active.mean()) if active.size else np.nan

    q_charge_ah = float(np.trapezoid(i_ch, t) / 3600.0) if np.isfinite(duration) and duration > 0 else np.nan
    e_charge_wh = float(np.trapezoid(v * i_ch, t) / 3600.0) if np.isfinite(duration) and duration > 0 else np.nan

    # CC/CV heuristic thresholds (tunable)
    time_to_4p0 = _time_to_voltage(t, v, 4.0, direction="above")
    time_to_4p2 = _time_to_voltage(t, v, 4.2, direction="above")
    cv_mask = v >= 4.18
    cv_time_s = float(t[cv_mask][-1] - t[cv_mask][0]) if cv_mask.any() else 0.0
    i_end = float(i_ch[-1]) if i_ch.size else np.nan
    v_end = float(v[-1]) if v.size else np.nan

    feats = {
        "filename": filename,
        "chg_duration_s": duration,
        "chg_frac_active": frac_active,
        "chg_time_to_4p0": time_to_4p0,
        "chg_time_to_4p2": time_to_4p2,
        "chg_cv_time_s": cv_time_s,
        "chg_q_ah": q_charge_ah,
        "chg_energy_wh": e_charge_wh,
        "chg_v_end": v_end,
        "chg_i_end": i_end,
        "chg_temp_rise": float(np.nanmax(temp) - np.nanmin(temp)) if np.isfinite(temp).any() else np.nan,
        "chg_temp_max": float(np.nanmax(temp)) if np.isfinite(temp).any() else np.nan,
    }
    feats.update(_series_stats(v, "chg_v"))
    feats.update(_series_stats(i_ch, "chg_i"))
    feats.update(_series_stats(temp, "chg_temp"))

    # If commanded columns exist, add tracking error features
    if "Current_charge" in df.columns:
        icmd = _to_num(df["Current_charge"])[m][order]
        feats["chg_i_cmd_mae"] = float(np.nanmean(np.abs(i_ch - np.maximum(icmd, 0.0))))
    if "Voltage_charge" in df.columns:
        vcmd = _to_num(df["Voltage_charge"])[m][order]
        feats["chg_v_cmd_mae"] = float(np.nanmean(np.abs(v - vcmd)))

    return feats

### Extracting features from discharge records to a new DataFrame

In [5]:
# Extract features from discharge + charge cycles, then build a modeling table
try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# Make sure types are normalized
metadata = metadata.copy()
metadata["type"] = metadata["type"].astype(str).str.strip().str.lower()
metadata["battery_id"] = metadata["battery_id"].astype(str).str.strip()
metadata["filename"] = metadata["filename"].astype(str).str.strip()
metadata["test_id"] = pd.to_numeric(metadata["test_id"], errors="coerce")

discharge_meta = metadata.loc[metadata["type"] == "discharge"].copy()
charge_meta = metadata.loc[metadata["type"] == "charge"].copy()

# --- Discharge features ---
dis_feats = []
for _, row in tqdm(discharge_df.iterrows(), total=len(discharge_df), desc="Discharge feature extraction"):
    fname = row['filename']
    current_capacity = row['Capacity']
    battery_id = row['battery_id']
    c_init = c_init_map.get(battery_id)
    feats = extract_discharge_features(fname, current_capacity, c_init)
    if feats is not None:
        dis_feats.append(feats)
dis_feats_df = pd.DataFrame(dis_feats)
discharge_enriched = discharge_meta.merge(dis_feats_df, on="filename", how="inner")

print(f"Discharge cycles with extracted features: {len(discharge_enriched)}")
display(discharge_enriched.head())

# --- Charge features ---
chg_feats = []
for _, row in tqdm(charge_meta.iterrows(), total=len(charge_meta), desc="Charge feature extraction"):
    feats = extract_charge_features(row["filename"])
    if feats is not None:
        chg_feats.append(feats)
chg_feats_df = pd.DataFrame(chg_feats)
charge_enriched = charge_meta.merge(chg_feats_df, on="filename", how="inner")

print(f"Charge cycles with extracted features: {len(charge_enriched)}")
display(charge_enriched.head())

# --- Align charge features to discharge rows (previous charge per battery) ---
charge_key = charge_enriched.sort_values(["battery_id", "test_id"])
dis_key = discharge_enriched.sort_values(["battery_id", "test_id"])

aligned_chunks = []
for bid, d in dis_key.groupby("battery_id", sort=False):
    c = charge_key.loc[charge_key["battery_id"] == bid]
    if c.empty:
        # no charge info for this battery
        aligned = d.copy()
        aligned_chunks.append(aligned)
        continue
    # merge_asof: match the last charge cycle at/before this discharge test_id
    aligned = pd.merge_asof(
        d.sort_values("test_id"),
        c.sort_values("test_id"),
        on="test_id",
        direction="backward",
        suffixes=("", "_chg"),
    )
    aligned_chunks.append(aligned)

model_table = pd.concat(aligned_chunks, ignore_index=True) if aligned_chunks else discharge_enriched.copy()

# Optional: bring impedance summary if you have it (cleaned impedance file)
if os.path.exists(impedance_final_path):
    imp = pd.read_csv(impedance_final_path)
    imp = imp[["battery_id", "test_id", "Re", "Rct"]]

    # 1. Clean IDs to ensure they match
    imp["battery_id"] = imp["battery_id"].astype(str).str.strip()
    imp["test_id"] = pd.to_numeric(imp["test_id"], errors="coerce")
    
    # Ensure model_table IDs are also compatible
    model_table["battery_id"] = model_table["battery_id"].astype(str).str.strip()
    model_table["test_id"] = pd.to_numeric(model_table["test_id"], errors="coerce")

    # 2. CRITICAL STEP: Both dataframes MUST be sorted by the key (test_id)
    imp = imp.sort_values("test_id")
    model_table = model_table.sort_values("test_id")

    # 3. Use merge_asof to "look ahead"
    # direction='forward' means: For Test 1, look forward to find the first match (Test 4)
    model_table = pd.merge_asof(
        model_table, 
        imp, 
        on="test_id", 
        by="battery_id", 
        direction="forward",
        suffixes=("", "_imp")
    )

    print("Merged impedance using cascading forward fill.")

else:
    print("No cleaned impedance metadata found at:", impedance_final_path)



Discharge feature extraction: 100%|██████████| 2249/2249 [00:30<00:00, 73.81it/s]


Discharge cycles with extracted features: 2247


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,...,temp_min,temp_mean,temp_std,v_t10,v_t50,v_t90,dvdt_min,dvdt_max,dvdt_mean,dvdt_std
0,discharge,2010-07-21 15:00:35.093,4,B0047,0,1,00001.csv,1.674305,NaN,NaN,...,5.008084,8.272423,1.453783,3.813664,3.490626,2.980600,-0.014905,0.034869,-0.000143,0.001816
1,discharge,2010-07-21 21:02:56.984,4,B0047,4,5,00005.csv,1.524366,NaN,NaN,...,5.454957,8.210715,1.238715,3.802706,3.505356,3.115401,-0.013276,0.036546,-0.000187,0.001973
2,discharge,2010-07-22 01:40:06.218,4,B0047,6,7,00007.csv,1.508076,NaN,NaN,...,4.922178,7.954455,1.415210,3.795278,3.503476,3.092692,-0.012701,0.037089,-0.000192,0.001999
3,discharge,2010-07-22 06:16:21.781,4,B0047,8,9,00009.csv,1.483558,NaN,NaN,...,4.553269,7.985865,1.376513,3.799047,3.483375,3.089111,-0.013379,0.038546,-0.000179,0.002096
4,discharge,2010-07-22 10:51:48.203,4,B0047,10,11,00011.csv,1.467139,NaN,NaN,...,4.826283,8.009427,1.321760,3.802117,3.480119,3.076176,-0.013272,0.038843,-0.000175,0.002116


Charge feature extraction: 100%|██████████| 2477/2477 [00:51<00:00, 48.40it/s]

Charge cycles with extracted features: 2477


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,...,chg_v_std,chg_i_min,chg_i_max,chg_i_mean,chg_i_std,chg_temp_min,chg_temp_mean,chg_temp_std,chg_i_cmd_mae,chg_v_cmd_mae
0,charge,2010-07-21 17:25:40.671,4,B0047,2,3,00003.csv,NaN,NaN,NaN,...,0.054005,0.001417,1.494314,0.520792,0.525546,4.358202,6.279191,1.080201,0.002923,0.319102
1,charge,2010-07-21 22:38:43.484,4,B0047,5,6,00006.csv,NaN,NaN,NaN,...,0.074499,0.000000,1.495911,0.518688,0.517788,4.161258,6.414351,1.269756,0.002900,0.328958
2,charge,2010-07-22 03:14:53.218,4,B0047,7,8,00008.csv,NaN,NaN,NaN,...,0.071711,0.000000,1.493423,0.513529,0.513147,3.948085,6.263927,1.227414,0.003007,0.317887
3,charge,2010-07-22 07:50:21.625,4,B0047,9,10,00010.csv,NaN,NaN,NaN,...,0.067625,0.000000,1.494114,0.507374,0.510143,4.021521,6.251470,1.224443,0.003024,0.301465
4,charge,2010-07-22 12:25:03.656,4,B0047,11,12,00012.csv,NaN,NaN,NaN,...,0.065565,0.000757,1.495550,0.501276,0.506546,3.614540,6.214979,1.347860,0.002997,0.293386


Merged impedance using cascading forward fill.


### Saving the new dataset (with new features):

In [6]:
cols_to_drop = [
    'type',          # Constant value ('discharge')
    'start_time',    # Redundant with test_id (Cycle)
    'uid',           # Arbitrary ID
    'filename',      # No longer needed after merge
    'Re',            # Empty for discharge
    'Rct'            # Empty for discharge
]

# 2. Drop them (errors='ignore' prevents crash if column already gone)
model_table = model_table.drop(columns=cols_to_drop, errors='ignore')

# 3. Rename 'test_id' to 'Cycle' for clarity
model_table = model_table.rename(columns={'test_id': 'Cycle'})

# Final Check
print(f"Final Data Shape: {model_table.shape}")
print(model_table.info())
# Save
model_table.to_csv(feature_aggregation_path, index=False)
print("Saved modeling feature table to:", feature_aggregation_path)
display(model_table.head())

Final Data Shape: (2247, 72)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2247 entries, 0 to 2246
Data columns (total 72 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ambient_temperature      2247 non-null   int64  
 1   battery_id               2247 non-null   object 
 2   Cycle                    2247 non-null   int64  
 3   Capacity                 2247 non-null   float64
 4   dis_time                 2247 non-null   float64
 5   duration_s               2247 non-null   float64
 6   frac_active_discharge    2247 non-null   float64
 7   time_to_3v6              2247 non-null   float64
 8   time_to_3v8              2247 non-null   float64
 9   volt_drop                2247 non-null   float64
 10  r0_ohm                   2247 non-null   float64
 11  volt_at_600s             2247 non-null   float64
 12  cap_int                  2247 non-null   float64
 13  energy_int_wh            2247 non-null   float64


,ambient_temperature,battery_id,Cycle,Capacity,dis_time,duration_s,frac_active_discharge,time_to_3v6,time_to_3v8,volt_drop,...,chg_i_max,chg_i_mean,chg_i_std,chg_temp_min,chg_temp_mean,chg_temp_std,chg_i_cmd_mae,chg_v_cmd_mae,Re_imp,Rct_imp
0,4,B0045,0,1.081979,6436.141,6436.141,0.675510,599.969,180.906,0.245601,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.072628,0.245653
1,4,B0048,0,1.657996,6436.141,6436.141,0.930612,1804.766,508.313,0.218139,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.077991,0.239749
2,4,B0046,0,1.728239,6436.141,6436.141,0.995918,2040.594,678.469,0.201407,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.091945,0.187072
3,4,B0047,0,1.674305,6436.141,6436.141,0.957143,2119.219,757.156,0.207434,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.056058,0.200970
4,24,B0005,1,1.856487,3690.234,3690.234,0.903553,1351.203,417.281,0.216621,...,1.514393,0.648627,0.603211,24.167062,25.324079,1.01131,0.005282,0.473015,0.044669,0.069456


## Remaining Useful Life (RUL):

In [7]:
# 1. Load the ground truth data
gt_df = pd.read_csv(feature_aggregation_path)

# Standardize IDs (String/Int consistency is a common pain point)
gt_df["battery_id"] = gt_df["battery_id"].astype(str).str.strip()
gt_df["Cycle"] = pd.to_numeric(gt_df["Cycle"], errors="coerce")

# 2. Calculate RUL
# We assume the last cycle in the file is the failure point (End of Life)
eol_cycles = gt_df.groupby("battery_id")["Cycle"].max().rename("Max_Cycle")

# Merge the Max_Cycle back to the dataframe
gt_df = gt_df.merge(eol_cycles, on="battery_id")

# Create the RUL column
gt_df["RUL"] = gt_df["Max_Cycle"] - gt_df["Cycle"]

# 3. Prepare the subset to merge
# We only want to bring in the targets: Capacity and RUL
targets_to_merge = gt_df[["battery_id", "Cycle", "Capacity", "RUL"]]

# 4. Merge into your main model_table
# Note: This assumes your model_table already has a 'Cycle' column. 
# If it only has 'test_id', you might need to extract Cycle from the filename first.
rul_model_table = model_table.merge(
    targets_to_merge, 
    on=["battery_id", "Cycle"], 
    how="left"
)
rul_model_table = rul_model_table.sort_values(by=['battery_id'])
print(f"Merged RUL and Capacity. Rows with valid targets: {rul_model_table['RUL'].notna().sum()}")
rul_model_table.to_csv(feature_aggregation_path, index=False)


Merged RUL and Capacity. Rows with valid targets: 2247
